# Molecular Melting Point Prediction using RDKit and Machine Learning

### About this Project

This project is my attempt to explore how **Machine Learning** can be used to predict the **melting point of molecules** from their chemical structure.

As someone interested in both **Chemistry** and **Machine Learning**, I wanted to work on a project that combines these two fields. Instead of using image or text datasets, this project focuses on molecular data represented as **SMILES** (Simplified Molecular Input Line Entry System).

Using the **RDKit** library, I will convert these molecular structures into numerical features (called molecular descriptors and fingerprints) that can be understood by Machine Learning models. I will then train different regression models and compare their performance in predicting melting points.

---

## What I Hope to Learn

Through this project, I aim to:

- Understand how molecular datasets are structured.
- Learn how RDKit represents molecules.
- Generate molecular descriptors and fingerprints.
- Build and evaluate Machine Learning regression models.
- Explore the connection between chemistry and data science.

---

## Project Workflow

1. Load and understand the dataset.
2. Clean and inspect the data.
3. Process SMILES using RDKit.
4. Generate molecular descriptors.
5. Train Machine Learning models.
6. Evaluate model performance.
7. Compare different approaches and summarize the results.

---

## Dataset

This project uses the **Jean-Claude Bradley Open Melting Point Dataset**, which contains experimentally measured melting points and molecular SMILES.


In [21]:
# Import necessary libraries

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Loading the Dataset
The objective is to:

- Load the Excel dataset.
- Check the dimensions of the dataset.
- View the first few rows.
- Understand the available columns before beginning data preprocessing.

In [22]:
dataset = pd.read_excel("BradleyMeltingPointDataset.xlsx")
dataset.head()

,key,name,smiles,mpC,csid,link,source,donotuse,donotusebecause
0,1,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,64018,http://www.alfa.com/en/GP100W.pgm?DSSTK=B24192,Alfa Aesar,NaN,NaN
1,2,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,403764,http://www.alfa.com/en/GP100W.pgm?DSSTK=A13073,Alfa Aesar,NaN,NaN
2,3,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,80080,http://www.alfa.com/en/GP100W.pgm?DSSTK=L15884,Alfa Aesar,NaN,NaN
3,4,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,63701,http://www.alfa.com/en/GP100W.pgm?DSSTK=B20252,Alfa Aesar,NaN,NaN
4,5,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,69388,http://www.alfa.com/en/GP100W.pgm?DSSTK=L08261,Alfa Aesar,NaN,NaN


In [23]:
dataset.shape

(28645, 9)

# 2. Data Cleaning (PRE OF Preprocessing)

Before training any Machine Learning model, it is important to clean the dataset. Real-world datasets often contain missing values, duplicate entries, invalid records, or information that is not useful for prediction.

In this phase, I will inspect the dataset and remove unnecessary or unreliable data while keeping the important information required for the project.

For this project, the two most important columns are:

- **name** – The common name of the molecule. This helps identify molecules while exploring the dataset.
- **smiles** – The molecular structure represented in SMILES format. This will be processed using RDKit to generate molecular features.
- **mpC** – The experimental melting point (in °C), which is the target variable for prediction.

The remaining columns contain identifiers, links, metadata, and quality information that are not required for model training. After cleaning, the dataset will be ready for molecular preprocessing and feature generation.

In [24]:
data = dataset[['name','smiles','mpC']]
data.head()

,name,smiles,mpC
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0


In [25]:
# lets see if we have a new null values or not
data.isnull().sum()

name      0
smiles    0
mpC       0
dtype: int64

looks good if we don't have null values in data we have

# 3. Processing SMILES with RDKit

Machine Learning models cannot directly understand chemical structures written as **SMILES (Simplified Molecular Input Line Entry System)**. Therefore, the first step is to convert these text representations into molecular objects using the **RDKit** library.

In this phase, I will:

- Convert each SMILES string into an RDKit molecule (`Mol`) object.
- Check for invalid or corrupted SMILES strings.
- Remove molecules that cannot be processed correctly.
- Prepare the dataset for molecular descriptor and fingerprint generation in the next phase.

By the end of this phase, every row in the dataset will contain a valid molecular structure that can be analyzed computationally.

In [26]:
import rdkit.Chem as Chem
from rdkit.Chem import Descriptors

In [27]:
data['mol'] = data['smiles'].apply(Chem.MolFromSmiles)

[00:20:35] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 11 12 13 15 16 17 18 19 22 23 24
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 7 8 10 11 12 13 15 16 17 19
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 6 7 8 9 11 12 13 15 16 17 18
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 11 12 13 15 16 17 18 19 21 22 23
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 11 12 13 15 16 17 18 19 20 21 22
[00:20:35] Explicit valence for atom # 20 C, 5, is greater than permitted
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 16 17 18 19 20 21 22 23 24
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 10 11 12 13 14 15 16 17 23
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 6
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8
[00:20:35] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 

as we observe in data we have certain abnormal points

In [28]:
# Count invalid molecules

invalid = data["mol"].isna().sum()

print(f"Invalid molecules: {invalid}")
print(f"Valid molecules  : {len(data) - invalid}")

Invalid molecules: 301
Valid molecules  : 28344


301 from 28645 its around 1.05% of data better messing up with this we should remove that fraction having data size of 28344 entries in it... still works good

In [29]:
data = data[data['mol'].notna()].reset_index(drop = True)

In [35]:
from rdkit.Chem.Draw import IPythonConsole

IPythonConsole.ipython_useSVG = True
IPythonConsole.drawOptions.addAtomIndices = False

data.head()

,name,smiles,mpC,mol
0,"2-(2,4-dinitrobenzyl)pyridine",c1ccnc(c1)Cc2ccc(cc2[N+](=O)[O-])[N+](=O)[O-],92.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5b60>
1,2-(1-piperidinyl)aniline,c1ccc(c(c1)N)N2CCCCC2,46.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5bd0>
2,2-(1-piperazinyl)pyrimidine,c1cnc(nc1)N2CCNCC2,33.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5c40>
3,2-(1-piperazinyl)phenol,c1ccc(c(c1)N2CCNCC2)O,125.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5cb0>
4,2-(1-cyclohexenyl)ethylamine,C1CCC(=CC1)CCN,-55.0,<rdkit.Chem.rdchem.Mol object at 0x76798e0d5d20>


In [31]:
data.shape

(28344, 4)